# Smoke detector (yolo11n) on the custom smoke dataset - train on Colab

ชุดข้อมูลสร้างด้วย `tools/build_smoke_dataset.py` (ScPuteri "Cigarette Vape Smoke" v11, CC BY 4.0, class เดียวคือ smoke)

**กฎ (ห้ามเปลี่ยน)**
- ใช้แค่ `dataset.zip` ที่สร้างจากสคริปต์ ห้ามใส่ภาพหรือคลิปจาก `~/smoke_eval_clips` หรือฉากทดสอบสด
- train/val แบ่งไว้แล้วตามกลุ่มภาพ ห้ามแบ่งใหม่ที่นี่
- ผล val ที่นี่**ไม่ใช่**ผลวัดของโปรเจกต์ ผลจริงต้องวัดด้วย `tools/eval_smoke_clips.py` และ `tools/smoke_live_test.py` บน Mac

**ก่อนรัน:** Runtime → Change runtime type → **T4 GPU** เวลาโดยประมาณ: 50 epoch ประมาณ 10–20 นาที


In [1]:
# 1. same Ultralytics as the Mac (8.4.136) and check the GPU
!pip install -q ultralytics==8.4.136
!nvidia-smi --query-gpu=name,memory.total --format=csv
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert ultralytics.__version__ == "8.4.136", "use the same version as the Mac"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 9.9 MB/s eta 0:00:00
name, memory.total [MiB]
Tesla T4, 15360 MiB
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics 8.4.136 | torch 2.11.0+cu130 | cuda True


In [3]:
# 2. dataset.zip from Google Drive -> local disk, then check it
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, shutil, json

ZIP = Path('/content/drive/MyDrive/smoke_custom/dataset.zip')
OUT_DIR = Path('/content/drive/MyDrive/smoke_custom/smoke_custom_yolo11n_v1')
WORK = Path('/content/smoke_custom')
assert 'smoke_eval_clips' not in str(ZIP), 'test clips must never be used for training'
assert ZIP.is_file(), f'{ZIP} not found'
if WORK.exists():
    shutil.rmtree(WORK)
with zipfile.ZipFile(ZIP) as z:
    bad = [n for n in z.namelist() if 'smoke_eval_clips' in n]
    assert not bad, f'zip contains test clips: {bad[:3]}'
    z.extractall(WORK)
DATA = next(WORK.rglob('data.yaml'))
ROOT = DATA.parent
print('data.yaml:', DATA, '\n' + DATA.read_text())
info = json.loads((ROOT / 'build_info.json').read_text())
print(json.dumps(info['report'], indent=2))
for split in ('train', 'val'):
    n_img = len(list((ROOT / split / 'images').iterdir())); n_lab = len(list((ROOT / split / 'labels').iterdir()))
    print(f'{split}: images {n_img} labels {n_lab}')
    assert n_img == n_lab == info['report'][split]['images'], 'dataset does not match build_info.json'
assert info['report']['near_duplicates_val_vs_train'] == 0


Mounted at /content/drive
data.yaml: /content/smoke_custom/dataset/data.yaml 
train: train/images
val: val/images
names:
  0: smoke

{
  "train": {
    "images": 1481,
    "smoke_images": 459,
    "negative_images": 1022,
    "extra_negative_images": 380,
    "smoke_boxes": 478,
    "groups": 805
  },
  "val": {
    "images": 184,
    "smoke_images": 63,
    "negative_images": 121,
    "extra_negative_images": 20,
    "smoke_boxes": 65,
    "groups": 133
  },
  "near_duplicates_val_vs_train": 0,
  "groups": 938,
  "images_moved_val_to_train_for_near_duplicates": 65
}
train: images 1481 labels 1481
val: images 184 labels 184


In [4]:
# 3. train yolo11n, smoke only (fixed settings)
from ultralytics import YOLO
ARGS = dict(data=str(DATA), epochs=50, imgsz=640, batch=32, seed=0, deterministic=True,
            patience=50, project='/content/runs', name='smoke_custom_yolo11n', exist_ok=True)
model = YOLO('yolo11n.pt')   # COCO-pretrained yolo11n from Ultralytics
model.train(**ARGS)


New https://pypi.org/project/ultralytics/8.4.173 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.136 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/smoke_custom/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f52a15975b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [5]:
# 4. save weights + record to Drive (download both to models/smoke/ on the Mac)
import datetime, hashlib
best = Path('/content/runs/smoke_custom_yolo11n/weights/best.pt')
m = YOLO(str(best))
val = m.val(data=str(DATA), split='val', imgsz=640)
OUT_DIR.mkdir(parents=True, exist_ok=True)
out = OUT_DIR / 'smoke_custom_yolo11n_v1.pt'
shutil.copy2(best, out)
record = {
    'file': out.name, 'sha256': hashlib.sha256(out.read_bytes()).hexdigest(),
    'date': datetime.datetime.now().isoformat(timespec='minutes'), 'base': 'yolo11n.pt',
    'classes': m.names, 'dataset_build_info': info,
    'train_args': {k: v for k, v in ARGS.items() if k != 'data'},
    'val_split_metrics': {'mAP50': float(val.box.map50), 'mAP50-95': float(val.box.map),
                          'precision': float(val.box.mp), 'recall': float(val.box.mr)},
    'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'note': 'val metrics are on the dataset split, not on our camera; measure with eval_smoke_clips / smoke_live_test',
}
(OUT_DIR / 'train_info.json').write_text(json.dumps(record, indent=2, ensure_ascii=False, default=str))
print(json.dumps(record['val_split_metrics'], indent=2), '\nsha256', record['sha256'])


Ultralytics 8.4.136 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1725.7±484.0 MB/s, size: 41.0 KB)
val: Scanning /content/smoke_custom/dataset/val/labels.cache... 184 images, 121 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 184/184 64.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 3.7it/s 3.2s0.2s
                   all        184         65      0.672      0.585      0.648      0.349
Speed: 3.1ms preprocess, 5.9ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /content/runs/detect/val
{
  "mAP50": 0.6482718390179492,
  "mAP50-95": 0.34903111901673844,
  "precision": 0.6719017923823475,
  "recall": 0.5846153846153846
} 
sha256 dee7fb2db1a38e2133fec43a48271aa4cb498c44fc5b08c1f296ca0d193b017d


## ขั้นต่อไปบน Mac

```bash
~/Desktop/nursery_guard_ai/venv/bin/python tools/smoke_live_test.py --model models/smoke/smoke_custom_yolo11n_v1.pt
```
เกณฑ์ A/B/C และ conf 0.25 เดิม ห้ามแก้ ห้ามต่อเข้า pipeline หลักจนกว่าจะสั่ง
